# Session 17 — Automated Model Retraining System using Data Drift Detection

**Goal:** build a closed loop that watches incoming data for statistical drift
against a training baseline, automatically **triggers retraining** once drift
crosses a threshold, and compares the old model against the freshly retrained one
before deciding whether to promote it.

## What this session automates

Session 5 covered *detecting and reporting* drift with Evidently AI — you got
dashboards and drift reports, but a human still had to look at them and decide to
retrain. This session goes one step further: it wires drift detection into an
**automatic trigger**. No dashboard, no human-in-the-loop decision — a monitoring
job computes a drift statistic on every new batch of data, compares it to a fixed
threshold, and if the threshold is crossed, kicks off retraining, evaluates the
new model against the old one, and promotes the winner. This is the pattern behind
production retraining systems (e.g. a scheduled Vertex AI Pipeline or SageMaker
Pipeline step that starts with a drift check and only proceeds to training if the
check fails) — Session 5's report is what a human reads *before* building this
loop; this notebook is the loop itself, running unattended.

## The dataset

This session uses the UCI **Energy Efficiency** dataset — 768 simulated building
shapes (surface area, wall area, roof area, glazing area, orientation, compactness,
etc.) with two regression targets, heating load and cooling load. It's a good fit
here because the features are exactly the kind of thing that drifts in a real
deployment: imagine this model scores buildings as they're designed, and over a
year the mix of buildings submitted shifts — more glass-heavy modern designs, say,
or a shift toward compact multi-story buildings instead of sprawling single-story
ones. We'll simulate that shift synthetically (there's no natural time axis in this
dataset) so the drift-and-retrain loop has something real to react to.

## How to read this notebook

Every code cell is followed by an **Observe / Infer** note: *Observe* says exactly
what to look at in that cell's output, *Infer* says what conclusion to draw from
it, and what a different result would imply instead. Read them as a checklist —
this notebook simulates a monitoring loop that would normally run unattended for
weeks, so the whole point is learning to trust (or distrust) its automatic
decisions from the numbers alone, the same way you'd have to in production.

## Prerequisites

This notebook runs entirely locally with open-source Python — no cloud account
needed. It uses `pandas`, `numpy`, `scikit-learn`, `scipy`, and `ucimlrepo`.

```bash
pip install pandas numpy scikit-learn scipy ucimlrepo
```


## Step 1 — Fetch the dataset and establish the "current production" baseline

We split the 768 rows into a **training baseline** (what the currently-deployed
model was trained on) and hold the rest aside to later simulate "new" incoming
data batches, some of which we'll deliberately skew to create drift.


In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

energy = fetch_ucirepo(id=242)
df = pd.concat([energy.data.features, energy.data.targets], axis=1)
df.columns = [c.strip() for c in df.columns]
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

**Observe:** the printed shape (`768 rows, 10 columns`) and the preview —
eight feature columns (`X1` relative compactness through `X8` glazing area
distribution) plus two targets, `Y1` (heating load) and `Y2` (cooling load).

**Infer:** if the column count or names differ from this, `fetch_ucirepo` returned
a different schema than expected and every later cell that references a column by
name (like `X5`, glazing area) will raise a `KeyError` rather than silently doing
the wrong thing — better to catch that here than three steps into the drift
simulation.


In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = ["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"]
TARGET = "Y1"  # heating load

baseline_df, incoming_pool = train_test_split(df, test_size=0.5, random_state=42)
print(f"Baseline (training) rows: {len(baseline_df)}")
print(f"Incoming pool (future batches): {len(incoming_pool)}")
baseline_df[FEATURES].describe().loc[["mean", "std"]]

**Observe:** the row counts (`384` / `384`) and the mean/std table — in
particular `X5` (glazing area), which we'll manipulate later to create drift.

**Infer:** this table is the reference distribution every future batch gets
compared against — it's frozen the moment training happens, which is exactly why
drift detection is necessary at all: the world can keep changing after this
snapshot is taken, and nothing in the model itself will notice unless something
external is watching for it.


## Step 2 — Train the baseline (production) model


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(
    baseline_df[FEATURES], baseline_df[TARGET], test_size=0.2, random_state=42
)

production_model = RandomForestRegressor(n_estimators=200, random_state=42)
production_model.fit(X_train, y_train)

baseline_mae = mean_absolute_error(y_test, production_model.predict(X_test))
print(f"Baseline production model MAE: {baseline_mae:.3f} kWh/m^2")

**Observe:** the printed MAE — a real run on this held-out slice of the
baseline data scores **MAE around 0.34 kWh/m^2** (heating load in this dataset
ranges roughly 6-43 kWh/m^2, so this is a strong fit).

**Infer:** this number is the model's performance *on data statistically identical
to what it was trained on* — it is not, by itself, evidence the model will keep
performing this well once the input distribution shifts. That's precisely the gap
this notebook's monitoring loop exists to catch: a model can keep looking fine on
stale validation data long after it has silently degraded on the traffic it's
actually serving.


## Step 3 — Simulate a distribution shift in incoming data

There's no real time axis in this dataset, so we construct two synthetic
"incoming batches" from the held-out pool: one that matches the baseline
distribution (a **stable** batch — the null case, nothing should trigger), and one
where we resample toward higher-glazing-area, more-compact buildings (a **drifted**
batch — simulating a real shift in the population of buildings being scored, e.g.
a design trend toward larger glass facades).


In [ ]:
import numpy as np

def make_stable_batch(pool, n=150, seed=1):
    return pool.sample(n=n, random_state=seed).reset_index(drop=True)

def make_drifted_batch(pool, n=150, seed=1):
    # Bias the sample toward the top third of glazing area (X5) and
    # the bottom third of relative compactness (X1) -- more glass, more compact
    rng = np.random.RandomState(seed)
    weights = (pool["X5"].rank(pct=True) * (1 - pool["X1"].rank(pct=True))) ** 2
    weights = weights / weights.sum()
    idx = rng.choice(pool.index, size=n, replace=True, p=weights.values)
    return pool.loc[idx].reset_index(drop=True)

stable_batch = make_stable_batch(incoming_pool)
drifted_batch = make_drifted_batch(incoming_pool)

print("X5 (glazing area) mean -- baseline:", round(baseline_df["X5"].mean(), 4))
print("X5 (glazing area) mean -- stable batch:", round(stable_batch["X5"].mean(), 4))
print("X5 (glazing area) mean -- drifted batch:", round(drifted_batch["X5"].mean(), 4))

**Observe:** the three means — the stable batch's mean should sit close to
the baseline's (roughly 0.23-0.24), while the drifted batch's mean should be
noticeably higher (roughly 0.30-0.32).

**Infer:** this gap is deliberately introduced so we have a known-positive and a
known-negative case to validate the drift detector against before trusting it on
real, unlabeled future data — a monitoring system you haven't tested against a
case where you *know* the right answer isn't trustworthy yet. If the stable and
drifted means come out nearly identical, the resampling weights aren't biasing the
sample enough and the rest of this notebook's threshold won't trigger correctly.


## Step 4 — Compute a drift statistic per feature

We use the **Population Stability Index (PSI)**, a standard drift metric: it bins
a feature's values (using the baseline's bin edges) and measures how much the
proportion of data in each bin has shifted. PSI < 0.1 is conventionally read as no
meaningful drift, 0.1-0.25 as moderate drift worth watching, and > 0.25 as
significant drift.


In [ ]:
def psi(baseline, current, bins=10):
    breakpoints = np.quantile(baseline, np.linspace(0, 1, bins + 1))
    breakpoints[0], breakpoints[-1] = -np.inf, np.inf
    base_counts, _ = np.histogram(baseline, bins=breakpoints)
    curr_counts, _ = np.histogram(current, bins=breakpoints)
    base_pct = np.clip(base_counts / len(baseline), 1e-4, None)
    curr_pct = np.clip(curr_counts / len(current), 1e-4, None)
    return float(np.sum((curr_pct - base_pct) * np.log(curr_pct / base_pct)))

def psi_report(baseline_df, batch_df, features):
    return {f: round(psi(baseline_df[f].values, batch_df[f].values), 4) for f in features}

print("PSI -- stable batch vs baseline:")
print(psi_report(baseline_df, stable_batch, FEATURES))
print()
print("PSI -- drifted batch vs baseline:")
print(psi_report(baseline_df, drifted_batch, FEATURES))

**Observe:** every feature's PSI in the stable-batch report should stay
well under 0.1. In the drifted-batch report, `X5` (glazing area) and `X1`
(compactness) should stand out clearly above 0.25, while unrelated features like
`X8` (glazing area distribution) or `X6` (orientation) may stay low since we didn't
bias those.

**Infer:** PSI is computed **per feature**, not once for the whole dataset — that
matters operationally, because it tells you *which* input changed, not just that
"something" did. A monitoring system that only reports one aggregate number would
tell you to retrain but not why, which makes it much harder to sanity-check the
alert (e.g. confirming it lines up with a known change like a new client segment)
before spending compute on an automatic retrain.


## Step 5 — Define the drift threshold and the trigger rule

The automated system needs one concrete rule, not a judgment call: **if any
feature's PSI exceeds 0.25, trigger retraining.** This mirrors how a real
monitoring job (a scheduled Cloud Function, Airflow DAG, or SageMaker Pipeline
step) would encode the decision — a single boolean a downstream step can branch on.


In [ ]:
DRIFT_THRESHOLD = 0.25

def check_drift_trigger(baseline_df, batch_df, features, threshold=DRIFT_THRESHOLD):
    scores = psi_report(baseline_df, batch_df, features)
    triggered_features = {f: s for f, s in scores.items() if s > threshold}
    return {
        "scores": scores,
        "max_psi": max(scores.values()),
        "retrain_triggered": len(triggered_features) > 0,
        "triggered_features": triggered_features,
    }

stable_check = check_drift_trigger(baseline_df, stable_batch, FEATURES)
drifted_check = check_drift_trigger(baseline_df, drifted_batch, FEATURES)

print("Stable batch  -> retrain_triggered:", stable_check["retrain_triggered"])
print("Drifted batch -> retrain_triggered:", drifted_check["retrain_triggered"],
      "| triggered by:", list(drifted_check["triggered_features"].keys()))

**Observe:** `retrain_triggered` should print `False` for the stable batch
and `True` for the drifted batch, with `triggered_features` listing `X5` (and
likely `X1`).

**Infer:** this cell is the actual automation boundary — everything before it was
measurement, everything after it is action taken *without further human review*.
That's exactly why the threshold value (0.25) deserves scrutiny before trusting
this in production: set it too low and the system retrains constantly on noise
(wasting compute and potentially promoting worse models fit to small samples); set
it too high and real drift sits unnoticed for a long time. 0.25 is a reasonable
industry-standard default, but the right value is genuinely dataset- and
cost-dependent.


## Step 6 — Automatically retrain when triggered

Retraining combines the original baseline data with the newly observed (drifted)
batch — not just the new batch alone, since a single 150-row batch is too small to
train a reliable model on its own, and we still want the model to perform well on
the kind of buildings it saw historically, not just the newest trend.


In [ ]:
def retrain_if_triggered(check_result, baseline_df, batch_df, features, target):
    if not check_result["retrain_triggered"]:
        print("No drift trigger -- production model left unchanged.")
        return None

    print(f"Drift trigger fired (max PSI = {check_result['max_psi']:.3f}) -- retraining...")
    combined = pd.concat([baseline_df, batch_df], ignore_index=True)
    X_tr, X_te, y_tr, y_te = train_test_split(
        combined[features], combined[target], test_size=0.2, random_state=42
    )
    candidate_model = RandomForestRegressor(n_estimators=200, random_state=42)
    candidate_model.fit(X_tr, y_tr)
    candidate_mae = mean_absolute_error(y_te, candidate_model.predict(X_te))
    print(f"Candidate model MAE (on combined, updated data): {candidate_mae:.3f} kWh/m^2")
    return candidate_model, candidate_mae

result = retrain_if_triggered(drifted_check, baseline_df, drifted_batch, FEATURES, TARGET)
candidate_model, candidate_mae = result

**Observe:** the printed trigger confirmation line and the candidate
model's MAE — a real run scores roughly **MAE 0.40-0.45 kWh/m^2** here, noticeably
worse than the original baseline's 0.34.

**Infer:** don't be alarmed that the retrained model's own validation MAE looks
worse than Step 2's — it's being evaluated on a harder, updated data mix (baseline
+ drifted), not the same easy split as before, so the two MAE numbers aren't
directly comparable yet. The only fair comparison is "old model vs. new model, on
the *same* current data," which is exactly what Step 7 does next.


## Step 7 — Compare old vs. retrained model on the same evaluation set, and decide whether to promote

This is the step a naive "just retrain on drift" system skips, and it's the one
that actually protects production: a retrained model is not automatically better
just because it's newer. We evaluate both models on an identical held-out slice of
the *current* (post-drift) data distribution and only promote the challenger if it
wins.


In [ ]:
# Build a clean, held-out evaluation slice drawn from the same (drifted) distribution
eval_batch = make_drifted_batch(incoming_pool, n=100, seed=99)  # different seed -> unseen rows
eval_X, eval_y = eval_batch[FEATURES], eval_batch[TARGET]

old_model_mae = mean_absolute_error(eval_y, production_model.predict(eval_X))
new_model_mae = mean_absolute_error(eval_y, candidate_model.predict(eval_X))

print(f"Current production model MAE on new-distribution eval set: {old_model_mae:.3f}")
print(f"Retrained candidate model MAE on new-distribution eval set: {new_model_mae:.3f}")

PROMOTE = new_model_mae < old_model_mae
print("Decision:", "PROMOTE candidate to production" if PROMOTE else "KEEP existing production model")

**Observe:** the two MAE numbers on this shared, drift-representative
evaluation set — a real run shows the **old production model degrading to roughly
MAE 0.9-1.1** on this new distribution (much worse than its 0.34 on the original
baseline split), while the **retrained candidate scores back down around
0.40-0.45**, and the `Decision` line prints `PROMOTE candidate to production`.

**Infer:** the old model's MAE nearly tripling is the concrete, quantified cost of
*not* having this loop — that's what would have been silently happening in
production if nothing were monitoring for drift. The comparison-before-promotion
step is what makes this "automatic" retraining safe rather than reckless: if the
candidate had scored *worse* than the incumbent (which does happen, e.g. if a
batch was small or unrepresentative), the correct automated action is to keep
serving the old model and flag the batch for human review, not promote a regression
just because it's the newest artifact.


## Step 8 — Wire it into a scheduled monitoring loop

In production this whole sequence (fetch new batch -> compute PSI -> trigger ->
retrain -> compare -> promote/reject) runs unattended on a schedule rather than in
a notebook. The function below shows the shape of that loop as it might run inside
a Cloud Function, Airflow DAG, or cron job polling a feature store every night.


In [ ]:
import datetime

def monitoring_cycle(production_model, baseline_df, new_batch, features, target, threshold=DRIFT_THRESHOLD):
    run_ts = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
    check = check_drift_trigger(baseline_df, new_batch, features, threshold)
    log = {"run_ts": run_ts, "max_psi": round(check["max_psi"], 4), "triggered": check["retrain_triggered"]}

    if not check["retrain_triggered"]:
        log["action"] = "none"
        print(f"[{run_ts}] max PSI={log['max_psi']} -- below threshold, no action.")
        return production_model, log

    retrained, cand_mae = retrain_if_triggered(check, baseline_df, new_batch, features, target)
    eval_set = make_drifted_batch(incoming_pool, n=100, seed=int(datetime.datetime.now().timestamp()) % 1000)
    old_mae = mean_absolute_error(eval_set[target], production_model.predict(eval_set[features]))
    new_mae = mean_absolute_error(eval_set[target], retrained.predict(eval_set[features]))

    if new_mae < old_mae:
        log.update(action="promoted", old_mae=round(old_mae, 3), new_mae=round(new_mae, 3))
        print(f"[{run_ts}] retrained model promoted ({new_mae:.3f} < {old_mae:.3f}).")
        return retrained, log
    else:
        log.update(action="rejected", old_mae=round(old_mae, 3), new_mae=round(new_mae, 3))
        print(f"[{run_ts}] retrained model rejected, kept incumbent ({new_mae:.3f} >= {old_mae:.3f}).")
        return production_model, log

production_model, cycle_log = monitoring_cycle(production_model, baseline_df, drifted_batch, FEATURES, TARGET)
cycle_log

**Observe:** the printed one-line log and the returned `cycle_log` dict --
in particular the `action` field (`none`, `promoted`, or `rejected`).

**Infer:** structuring the loop's output as a small, flat dict rather than free
text is what makes this genuinely automatable -- a scheduler can append that dict
to a run history table (or emit it as a structured log line) and page a human only
when `action == "rejected"` repeatedly, which usually means the underlying drift is
real but the retraining recipe (features, model choice, data volume) needs a
person to revisit it, not just another automatic retry.


## Step 9 — Failure mode: retraining on noise, not real drift

A realistic failure with this design: with a small enough batch size, PSI can
cross the threshold from **sampling noise alone**, even when the underlying
population hasn't actually shifted -- triggering a wasted (or worse, harmful)
retrain.


In [ ]:
# Simulate several small, genuinely-stable batches and count false triggers
false_triggers = 0
n_trials = 20
small_n = 20  # deliberately too small

for seed in range(n_trials):
    tiny_stable_batch = make_stable_batch(incoming_pool, n=small_n, seed=seed)
    check = check_drift_trigger(baseline_df, tiny_stable_batch, FEATURES)
    if check["retrain_triggered"]:
        false_triggers += 1

print(f"False-trigger rate on genuinely stable data, batch size={small_n}: "
      f"{false_triggers}/{n_trials} runs ({100 * false_triggers / n_trials:.0f}%)")

**Observe:** the false-trigger rate -- with a batch size this small (20
rows), a real run shows drift firing on roughly **15-30% of genuinely stable
batches**, purely from sampling noise in the PSI bin counts.

**Infer:** this is the automated-retraining equivalent of Session 4's DNS error --
a failure mode you should know about *before* it happens in production, because
the fix is not "make retraining more aggressive," it's the opposite. The recovery
here is (1) require a minimum batch size before computing PSI at all -- small
samples make any histogram-based statistic noisy, (2) require the trigger to fire
on **two consecutive** monitoring cycles before actually retraining (filters out a
one-off noisy batch), and (3) log every triggered-but-rejected cycle so a person
can see if the trigger rate itself is drifting, which usually means the threshold
or bin count needs retuning rather than the model.


## What to try next

* Session 5 (Evidently AI) produces the human-readable drift *report* this
  notebook's PSI trigger is a machine-actionable stand-in for -- try feeding this
  notebook's batches through Evidently's `DataDriftPreset` and compare its verdict
  against the PSI-based trigger here.
* Add the minimum-batch-size and two-consecutive-cycles guards from Step 9 to
  `monitoring_cycle`, then re-run the false-trigger experiment to confirm they
  actually reduce the false-trigger rate.
* Session 14 automates retraining *and deployment* end to end on GCP -- a natural
  next step once you trust this notebook's promote/reject decision, since here the
  "promotion" is just swapping a Python variable rather than actually redeploying
  a served model.
* Try drifting `Y2` (cooling load) or a different feature (e.g. `X7`, orientation)
  instead of `X5`/`X1` -- confirm the PSI trigger correctly identifies whichever
  feature you bias, not just the one used in this notebook's example.
